# Build email extractor

In [21]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, Literal
from constants import MODEL_LARGE
from pydantic_ai import Agent


class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_categry: Literal["damaged_product", "technical", "billing", "other"]
    urgency: Literal["high", "medium", "low"]
    summary: str = Field(description="Summarize the email in 3-4 sentences")

email_extractor_agent = Agent(
    "openrouter:nvidia/nemotron-3-super-120b-a12b:free",
    system_prompt=""" 
You are a customer support agent, your task is to
extract relevant information from an email.
""", output_type=EmailExtractor)


TODO: load in the data and test the agent on one email

In [2]:
import pandas as pd

df = pd.read_json("emails_cleaned.json")
df 


,inputs,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea..."


In [3]:
sample_mail = df.iloc[2]["inputs"]["email"]
sample_mail

"From: Oscar Johansson <oscar.johansson@yahoo.se>\nSubject: Cannot access my account for 3 days - Urgent help needed\n\nHello Support Team,\n\nI am reaching out because I have been completely locked out of my account for the past three days and I am running out of ideas on how to fix this on my own. The problem started on Monday evening when I tried to log in as usual but kept receiving an 'Invalid credentials' error despite being absolutely certain that I was entering the correct password.\n\nI followed the instructions on your website to reset my password, but the problem is that the password reset email never arrives in my inbox. I have checked my spam and junk folders multiple times, and there is nothing there either. I have attempted the reset process at least six or seven times across different browsers and even from my phone, but the result is always the same - no email arrives.\n\nThis is causing me real problems because I have important documents and data stored in my account 

In [4]:
result = await email_extractor_agent.run(sample_mail)
result

AgentRunResult(output=EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_categry='technical', urgency='high', summary='Oscar Johansson has been locked out of his account for three days due to invalid credentials. Password reset attempts fail because reset emails are not arriving in his inbox or spam folder, despite multiple tries across browsers and devices. He urgently needs access to important work documents before a Friday deadline and requests identity verification to resolve the issue.'))

In [5]:
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_categry='technical', urgency='high', summary='Oscar Johansson has been locked out of his account for three days due to invalid credentials. Password reset attempts fail because reset emails are not arriving in his inbox or spam folder, despite multiple tries across browsers and devices. He urgently needs access to important work documents before a Friday deadline and requests identity verification to resolve the issue.')

## Load in prompts from MLFlow

In [8]:
from mlflow import load_prompt


class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"] = Field(
        description=load_prompt("email-urgency-description").format()
    )
    summary: str = Field(
        description=load_prompt("summary-description").format(num_sentences=4))


email_extractor_agent = Agent(
    "openrouter:nvidia/nemotron-3-super-120b-a12b:free",
    system_prompt=load_prompt("email-extractor-system-prompt").format(),
    output_type=EmailExtractor,
)

/Users/kevinbruno/Documents/mlops/mlops-molnplattformar/llmops_kevin_bruno_mlo25/.venv/lib/python3.12/site-packages/mlflow/prompt/registry_utils.py:209: FutureWarning: The `mlflow.load_prompt` API is moved to the `mlflow.genai` namespace. Please use `mlflow.genai.load_prompt` instead. The original API will be removed in the future release.
  return func(*args, **kwargs)


In [9]:
result = await email_extractor_agent.run(sample_mail)

In [10]:
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johansson has been completely locked out of his account for the past three days due to invalid credentials despite entering the correct password. He has attempted to reset his password multiple times, but the reset emails never arrive in his Yahoo inbox or spam/junk folders. This is preventing him from accessing important work documents needed to meet an upcoming Friday deadline. He requests urgent assistance and offers to verify his identity to resolve the issue.')

## LLM judge

In [11]:
result.output.model_dump()

{'sender_name': 'Oscar Johansson',
 'sender_email': 'oscar.johansson@yahoo.se',
 'issue_category': 'technical',
 'urgency': 'high',
 'summary': 'Oscar Johansson has been completely locked out of his account for the past three days due to invalid credentials despite entering the correct password. He has attempted to reset his password multiple times, but the reset emails never arrive in his Yahoo inbox or spam/junk folders. This is preventing him from accessing important work documents needed to meet an upcoming Friday deadline. He requests urgent assistance and offers to verify his identity to resolve the issue.'}

TODO: dataframe with inputs, expectations, outputs but only 1 row

In [14]:
df["outputs"] =[{},{}, result.output.model_dump(), {}]
df

,inputs,expectations,outputs
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L...",{}
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B...",{}
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea...",{}


In [16]:
df_sample = df.drop([0,1,3])
df_sample

,inputs,expectations,outputs
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."


In [18]:
from mlflow.genai.scorers import get_all_scorers

# get_all_scorers()

In [24]:
from mlflow.genai.scorers import Correctness, Summarization, Completeness, Fluency
import mlflow

llm_judge ="openrouter:/nvidia/nemotron-3-nano-30b-a3b:free" 

with mlflow.start_run(run_name="email-extractor-evalutation"):
    mlflow.log_param("model_judge", llm_judge)
    mlflow.log_param("model", MODEL_LARGE)

    results = mlflow.genai.evaluate(
        data = df_sample,
        scorers=[
            Correctness(model=llm_judge),
            Summarization(model=llm_judge)
        ]
    )

results

Evaluating: 100%|██████████| 1/1 [Elapsed: 00:03, Remaining: 00:00] [predict_fn: 0%, scorers: 100%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: email-extractor-evalutation
  Run ID: a9f9bd0717294e58ad9727e9d7d8352d

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



EvaluationResult(
  run_id: a9f9bd0717294e58ad9727e9d7d8352d
  metrics:
    summarization/mean: 1.0
    correctness/mean: 1.0
  result_df: 1 rows x 15 cols
)

In [25]:
results.result_df

,trace_id,summarization/value,correctness/value,expected_response/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-e0bdc8a4f51ab6c4c7129d54877d67ab,yes,yes,"{""sender_name"": ""Oscar Johansson"", ""sender_ema...","{""info"": {""trace_id"": ""tr-e0bdc8a4f51ab6c4c712...",None,OK,1776258703925,0,{'email': 'From: Oscar Johansson <oscar.johans...,"{'sender_name': 'Oscar Johansson', 'sender_ema...","{'mlflow.user': 'kevinbruno', 'mlflow.source.n...",{'mlflow.artifactLocation': '/Users/kevinbruno...,"[{'trace_id': '4L3IpPUatsTHEp1Uh31nqw==', 'spa...",[{'assessment_id': 'a-0bc310397f3046429b4b93b0...
